# Week 10 lab · R&B or rap? Check the model

**LDSCI6253 Advanced Information Presentation & Visualisation** · Northeastern University London

In the lecture a model guessed six genres from ten numbers, and five charts showed what it saw and where it went
wrong. Today: **one question, the same five checks.** You can hear the difference between R&B and rap. Can a model
tell them apart from the numbers alone, and how would you say so honestly?

| Part | You will… | The chart | Time |
|---|---|---|---|
| **A** | look first: how many songs, how many repeats, which labels disagree | counts | 8 min |
| **B** | find the columns where the two genres differ | box plots, repeated | 10 min |
| **C** | train a model and score it against the lazy answer | | 8 min |
| **D** | find where it is wrong, and what it listens to | confusion matrix, importance | 14 min |
| **E** | find out if it is learning or memorising | the depth curve | 10 min |
| **F** | hand in: the honest chart and its headline | the honest comparison | 10 min |

**How to work, in pairs.** One of you types, the other checks. Before each Run, the checker says what they expect
to see; after it, compare. Swap at each letter. The lecture's code is in `10_Notebook_IPDV_Machine_Learning.ipynb`
and `week10_build_its.py`: each part says which section to look at.

- Where you see `___`, replace it with code, then run the cell (Shift + Enter). Run cells **in order**.
- A ✅ or ❌ line after a cell says whether you are on track.
- Stuck? Open the 💡 **Hint**, then the tutorial section named.

**If you get an error**
- `NameError`: a cell above was not run, or a name is missing its quotes.
- An error that mentions **`BLANK`** or `___`: a blank is still empty.
- `MaxRowsError`: Altair draws up to 5,000 rows. Chart `sample`, not `unique`.
- `SchemaValidationError`: a column name is misspelt, or missing its type (`'speechiness:Q'`).

## 0 · Set up

Run this cell first. It loads the songs and sets the ten audio numbers a model sees.

In [ ]:
import altair as alt
import pandas as pd
from pathlib import Path

LIVE = "https://slideforge-rpjj.onrender.com/assets/lesson/ipdv/week10/"

def load(name):
    """Read name.csv from beside this notebook, or from the course site if it is not here."""
    here = Path(name + ".csv")
    return pd.read_csv(here if here.exists() else LIVE + name + ".csv")

def check(ok, good, hint):
    """Print ✅ and what you found, or ❌ and what to look at."""
    print('✅ ' + good if ok else '❌ Not yet: ' + hint)

___ = 'BLANK: replace ___ with your code'   # an empty blank names itself in the error

songs = load('songs')
FEATURES = ['danceability', 'energy', 'loudness',
            'speechiness', 'acousticness', 'instrumentalness',
            'liveness', 'valence', 'tempo', 'duration_ms']
pair = alt.Color('playlist_genre:N', title='genre', scale=alt.Scale(
    domain=['r&b', 'rap'], range=['#009e73', '#d55e00']))
songs.head()

## A · Look first (8 min · tutorial §1)

**A1.** Keep only the rows whose `playlist_genre` is R&B or rap. The genres are written `'r&b'` and `'rap'`.

In [ ]:
two = songs[songs['playlist_genre'].isin(___)]
len(two)

**A2.** The same song can sit in several playlists. Count the songs filed under **both** genres.

In [ ]:
genres_per_song = two.groupby('track_id')['playlist_genre'].nunique()
both = (genres_per_song > ___).sum()
check(both == 201, f'{both} songs are filed as both R&B and rap: their label is a coin toss',
      'count the songs with more than one genre')

**A3.** Keep one row per song, then shuffle (the file is sorted by playlist). Take a sample to chart.

In [ ]:
unique = two.drop_duplicates(___).sample(frac=1, random_state=0)
sample = unique.sample(4000, random_state=0)
check(len(unique) == 10423, f'{len(unique)} songs: {(unique.playlist_genre == "rap").sum()} rap, '
      f'{(unique.playlist_genre == "r&b").sum()} R&B', 'drop the repeats by their id')

✏️ *In one sentence: what does A2 mean for any model trained on these labels?*

## B · Which columns differ? (10 min · tutorial §1c and §2b)

**B1.** Draw a box plot per genre for four columns at once: one chart, repeated.

<details><summary>💡 <b>Hint</b></summary>

`mark_boxplot()` draws the boxes. `alt.repeat('column')` stands in for each name in the list.
</details>

In [ ]:
alt.Chart(sample).mark____(size=40).encode(
    x=alt.X('playlist_genre:N', title=None),
    y=alt.Y(alt.repeat('column'), type='quantitative'),
    color=pair.legend(None),
).properties(width=150, height=250).repeat(
    column=['speechiness', 'danceability', 'energy', 'tempo'],
)

✏️ *Which column will help a model most? Which will hardly help? Write your guess: Part D checks it.*

## C · Train and score (8 min · tutorial §3b)

**C1.** Hide one song in four from the model, so you can test it on songs it never saw. `stratify=y` keeps the same
share of each genre in both halves.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X, y = unique[FEATURES], unique['playlist_genre']
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=___, random_state=0, stratify=___)
model = RandomForestClassifier(300, random_state=0)
model.fit(X_tr, y_tr)
guess = model.predict(X_te)

**C2.** How often is it right, and what would the lazy answer score: always the commoner genre?

In [ ]:
accuracy = (guess == y_te).mean()
lazy = y_te.value_counts(normalize=True).___
check(abs(accuracy - 0.781) < 0.01 and abs(lazy - 0.526) < 0.01,
      f'the model: {accuracy:.0%}; the lazy answer: {lazy:.0%}', 'is random_state=0 there, and stratify=y?')

## D · Where is it wrong, and what does it listen to? (14 min · tutorial §4b and §3b)

**D1.** A confusion matrix as shares of each real genre: each row adds up to 100%.

<details><summary>💡 <b>Hint</b></summary>

Divide each row by its own total: `counts.sum(axis=1, keepdims=True)`. Cells are `mark_rect`.
</details>

In [ ]:
from sklearn.metrics import confusion_matrix

labels = ['r&b', 'rap']
counts = confusion_matrix(y_te, guess, labels=labels)
share = counts / counts.sum(axis=___, keepdims=True)
cm = (pd.DataFrame(share, index=labels, columns=labels)
        .reset_index(names='true')
        .melt('true', var_name='guessed', value_name='share'))

base = alt.Chart(cm).encode(
    x=alt.X('guessed:N', title='the model guessed'),
    y=alt.Y('true:N', title='the real genre'))
cells = base.mark____().encode(
    color=alt.Color('share:Q', scale=alt.Scale(scheme='blues', domain=[0, 1]), legend=None))
words = base.mark_text(fontSize=20).encode(
    text=alt.Text('share:Q', format='.0%'),
    color=alt.when(alt.datum.share > 0.5).then(alt.value('white')).otherwise(alt.value('black')))
(cells + words).properties(width=260, height=260)

**D2.** Shuffle each column in turn on the test songs: how much accuracy does the model lose?

In [ ]:
from sklearn.inspection import permutation_importance

shuffled = permutation_importance(model, X_te, y_te, n_repeats=5, random_state=0)
scores = pd.DataFrame({'feature': FEATURES, 'lost': shuffled.importances_mean})

alt.Chart(scores).mark_bar(color='#0c3354').encode(
    x=alt.X('lost:Q', axis=alt.Axis(format='%'), title='accuracy lost when shuffled'),
    y=alt.Y('feature:N', sort=___, title=None),
).properties(width=420, height=260)

✏️ *Was your guess from B right? Does speechiness *make* a song rap? Say why not in one sentence.*

## E · Learning, or memorising? (10 min · tutorial §4c)

**E1.** One tree, allowed to ask more and more questions (`max_depth`). Score it on the songs it learnt from and on
songs it never saw.

In [ ]:
from sklearn.model_selection import validation_curve
from sklearn.tree import DecisionTreeClassifier

depths = list(range(1, 21))
train, test = validation_curve(
    DecisionTreeClassifier(random_state=0), X, y,
    param_name=___, param_range=depths, cv=3)
curve = pd.DataFrame({'depth': depths,
    'songs it learnt from': train.mean(axis=1),
    'songs it never saw': test.mean(axis=1),
}).melt('depth', var_name='scored on', value_name='accuracy')

alt.Chart(curve).mark_line(point=True).encode(
    x=alt.X('depth:Q', title='how deep the tree may grow'),
    y=alt.Y('accuracy:Q', axis=alt.Axis(format='%'), scale=alt.Scale(zero=False)),
    color=alt.Color('scored on:N', scale=alt.Scale(range=['#9aa3ad', '#c8102e'])),
).properties(width=420, height=260)

**E2.** Find the best depth in code.

In [ ]:
new = curve[curve['scored on'] == 'songs it never saw']
best = new.loc[new['accuracy'].idxmax(), ___]
check(best == 9, f'best on new songs at depth {best}', "the row with the highest accuracy: its 'depth'")

## F · Tell the public (10 min · tutorial §5) and hand in

**F1.** One honest chart: the model, the lazy answer, and how often each genre is recognised, on one axis from 0 to
100%. Then **write its title as a finding** (Week 9): what should a reader take away?

In [ ]:
recall = {g: ((guess == g) & (y_te == g)).sum() / (y_te == g).sum() for g in labels}
honest = pd.DataFrame({'claim': [
        'the model is right', 'always saying "rap" is right',
        'real rap songs it recognises', 'real R&B songs it recognises'],
    'share': [accuracy, lazy, recall['rap'], recall['r&b']]})

finding = ___

bars = alt.Chart(honest).mark_bar(color='#0c3354').encode(
    x=alt.X('share:Q', axis=alt.Axis(format='%'), scale=alt.Scale(domain=[0, 1]), title=None),
    y=alt.Y('claim:N', sort=None, title=None))
labels_on = bars.mark_text(align='left', dx=6).encode(text=alt.Text('share:Q', format='.0%'))
(bars + labels_on).properties(width=420, height=170,
    title=alt.Title(finding, subtitle='Songs the model had never seen; the genre is the playlist’s'))

**F2 · Hand in** (Canvas, by midday on the Friday after the lab): this notebook, run top to bottom, and under this
cell **one sentence per part** (A to F) saying what its chart or number shows. For F, add: *what would a reader need
to know to disagree with your title?*

✏️ *A: … B: … C: … D: … E: … F: …*